<a href="https://colab.research.google.com/github/thasedasaidani/dvf-pipeline/blob/main/Analys_immobilier_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Comparaison de modèles de RÉGRESSION (le prix est un nombre, pas une catégorie)
## Comparaison des modèles de régression

#Les premiers essais utilisaient des modèles de classification, ce qui ne convient pas pour prédire
#un prix (c'est un nombre, pas une catégorie). Ici, je compare des modèles de régression à
#une référence simple. Le meilleur est l'arbre de décision (R² = 0,45, erreur moyenne de 407 €/m²)
#  c'est celui utilisé dans l'application.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# données : les 4 variables utilisées par l'application
df = pd.read_csv('/content/prix-immobilier-communes-dvf.csv')
y = df["prix_m2_median"]
X_2 = df[["fiabilite", "nom_departement", "prix_m2_median_departement", "nb_ventes"]]
X_2_encoded = pd.get_dummies(X_2, columns=["fiabilite", "nom_departement"], dtype=int)

# 80 % pour entraîner, 20 % pour tester
X_train, X_test, y_train, y_test = train_test_split(X_2_encoded, y, test_size=0.2, random_state=42)
mediane = X_train.median()
X_train, X_test = X_train.fillna(mediane), X_test.fillna(mediane)

modeles = {
    "Référence (prix du département)": None,
    "Régression linéaire": LinearRegression(),
    "Arbre de décision": DecisionTreeRegressor(max_depth=4, random_state=42),
    "Forêt aléatoire": RandomForestRegressor(n_estimators=200, random_state=42),
}
for nom, modele in modeles.items():
    if modele is None:
        pred = X_test["prix_m2_median_departement"]
    else:
        modele.fit(X_train, y_train)
        pred = modele.predict(X_test)
    print(f"{nom:35s} R² = {r2_score(y_test, pred):.2f}   erreur moyenne = {mean_absolute_error(y_test, pred):.0f} €/m²")

Référence (prix du département)     R² = -1.11   erreur moyenne = 813 €/m²
Régression linéaire                 R² = 0.42   erreur moyenne = 413 €/m²
Arbre de décision                   R² = 0.45   erreur moyenne = 407 €/m²
Forêt aléatoire                     R² = 0.32   erreur moyenne = 447 €/m²
